In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:

class UberDataAnalyzer:
    """
    Encapsulates all data loading, cleaning, analysis, and visualization
    logic for the Uber Data Analysis App.
    """

    def __init__(self):
        self.df = None                # Raw / cleaned dataframe
        self.cleaned = False          # Flag: has cleaning been performed?
        self.datetime_col = None      # Name of the detected datetime column
        self.insights = {}            # Stores computed insights for the summary report

In [ ]:

    def load_data(self, filepath):
        """
        Load the Uber dataset from a CSV file.
        Handles file-not-found and parsing errors gracefully.
        """
        try:
            if not os.path.exists(filepath):
                print(f"\n[ERROR] File not found: '{filepath}'")
                return False

            self.df = pd.read_csv(filepath)

            if self.df.empty:
                print("\n[ERROR] The CSV file is empty.")
                self.df = None
                return False

            print(f"\n[SUCCESS] Dataset loaded successfully from '{filepath}'")
            print(f"Rows: {self.df.shape[0]}, Columns: {self.df.shape[1]}")

            # Try to auto-detect a datetime-like column
            self._detect_datetime_column()
            return True

        except pd.errors.ParserError:
            print("\n[ERROR] Failed to parse the CSV file. Please check its format.")
            return False
        except Exception as e:
            print(f"\n[ERROR] An unexpected error occurred while loading data: {e}")
            return False

    def _detect_datetime_column(self):
        """
        Attempts to automatically detect which column contains datetime info,
        based on common naming patterns used in Uber datasets.
        """
        candidates = [
            "Date/Time", "date/time", "pickup_datetime", "Pickup_datetime",
            "datetime", "Datetime", "Date", "date", "trip_start_time"
        ]
        for col in candidates:
            if col in self.df.columns:
                self.datetime_col = col
                return

        # Fallback: search for any column with 'date' or 'time' in its name
        for col in self.df.columns:
            if "date" in col.lower() or "time" in col.lower():
                self.datetime_col = col
                return

        self.datetime_col = None  # Not found; user will be informed during cleaning

In [ ]:
def clean_data(self):
        """
        Perform data cleaning steps:
        - Handle missing values
        - Remove duplicate records
        - Convert date/time column into proper datetime format
        - Extract Month, Day, Hour, Weekday features for analysis
        """
        if self.df is None:
            print("\n[ERROR] No dataset loaded. Please load a dataset first.")
            return

        try:
            initial_rows = self.df.shape[0]

            # --- Handle missing values ---
            missing_before = self.df.isnull().sum().sum()
            # Drop rows where the datetime column itself is missing (critical for analysis)
            if self.datetime_col and self.datetime_col in self.df.columns:
                self.df = self.df.dropna(subset=[self.datetime_col])
            # For other columns, fill numeric NaNs with median, categorical with mode
            for col in self.df.columns:
                if self.df[col].isnull().any():
                    if pd.api.types.is_numeric_dtype(self.df[col]):
                        self.df[col] = self.df[col].fillna(self.df[col].median())
                    else:
                        if not self.df[col].mode().empty:
                            self.df[col] = self.df[col].fillna(self.df[col].mode()[0])
            missing_after = self.df.isnull().sum().sum()

            # --- Remove duplicate records ---
            duplicates_found = self.df.duplicated().sum()
            self.df = self.df.drop_duplicates()

            # --- Convert datetime column ---
            if self.datetime_col and self.datetime_col in self.df.columns:
                self.df[self.datetime_col] = pd.to_datetime(
                    self.df[self.datetime_col], errors="coerce"
                )
                # Drop rows where conversion failed
                self.df = self.df.dropna(subset=[self.datetime_col])

                # Extract useful time-based features
                self.df["Month"] = self.df[self.datetime_col].dt.month
                self.df["MonthName"] = self.df[self.datetime_col].dt.month_name()
                self.df["DayOfWeek"] = self.df[self.datetime_col].dt.day_name()
                self.df["Hour"] = self.df[self.datetime_col].dt.hour
            else:
                print("\n[WARNING] No datetime column detected. "
                      "Time-based analysis features will not be available.")

            final_rows = self.df.shape[0]
            self.cleaned = True

            # --- Cleaning summary ---
            print("\n--- DATA CLEANING SUMMARY ---")
            print(f"Initial rows           : {initial_rows}")
            print(f"Missing values before   : {missing_before}")
            print(f"Missing values after    : {missing_after}")
            print(f"Duplicate rows removed   : {duplicates_found}")
            print(f"Final rows after cleaning: {final_rows}")
            print("Data cleaning completed successfully.\n")

        except Exception as e:
            print(f"\n[ERROR] An error occurred during data cleaning: {e}")


In [ ]:
 def show_dataset_info(self):
        """Display number of rows/columns, data types, and summary statistics."""
        if self.df is None:
            print("\n[ERROR] No dataset loaded. Please load a dataset first.")
            return

        print("\n--- DATASET INFORMATION ---")
        print(f"Number of rows    : {self.df.shape[0]}")
        print(f"Number of columns : {self.df.shape[1]}")

        print("\nData Types:")
        print(self.df.dtypes)

        print("\nSummary Statistics:")
        try:
            print(self.df.describe(include="all").transpose())
        except Exception as e:
            print(f"[WARNING] Could not generate full summary statistics: {e}")


In [ ]:
      def analyze_by_month(self):
        if not self._check_ready():
            return

        month_counts = self.df["MonthName"].value_counts()
        # Sort by calendar order for readability
        month_order = ["January", "February", "March", "April", "May", "June",
                        "July", "August", "September", "October", "November", "December"]
        month_counts = month_counts.reindex(
            [m for m in month_order if m in month_counts.index]
        )

        print("\n--- TRIPS BY MONTH ---")
        print(month_counts)

        busiest_month = month_counts.idxmax()
        self.insights["busiest_month"] = busiest_month
        self.insights["busiest_month_count"] = int(month_counts.max())
        print(f"\nMost active month: {busiest_month} ({month_counts.max()} trips)")

In [ ]:
 def analyze_by_day(self):
        """Analyze and display the number of trips per day of the week."""
        if not self._check_ready():
            return

        day_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
                     "Friday", "Saturday", "Sunday"]
        day_counts = self.df["DayOfWeek"].value_counts().reindex(day_order)

        print("\n--- TRIPS BY DAY OF WEEK ---")
        print(day_counts)

        busiest_day = day_counts.idxmax()
        self.insights["busiest_day"] = busiest_day
        self.insights["busiest_day_count"] = int(day_counts.max())
        print(f"\nMost active day: {busiest_day} ({day_counts.max()} trips)")

In [ ]:
  def analyze_by_hour(self):
        """Analyze and display the number of trips per hour of the day."""
        if not self._check_ready():
            return

        hour_counts = self.df["Hour"].value_counts().sort_index()

        print("\n--- TRIPS BY HOUR OF DAY ---")
        print(hour_counts)

        peak_hour = int(hour_counts.idxmax())
        self.insights["peak_hour"] = peak_hour
        self.insights["peak_hour_count"] = int(hour_counts.max())
        print(f"\nPeak booking hour: {peak_hour}:00 ({hour_counts.max()} trips)")

In [ ]:
def generate_visualizations(self):
        """Generate all required charts: bar charts, histogram, and heatmap."""
        if not self._check_ready():
            return

        try:
            sns.set_style("whitegrid")

            # --- 1. Bar chart: Trips by Month ---
            month_order = ["January", "February", "March", "April", "May", "June",
                            "July", "August", "September", "October", "November", "December"]
            month_counts = self.df["MonthName"].value_counts().reindex(
                [m for m in month_order if m in self.df["MonthName"].unique()]
            )
            plt.figure(figsize=(10, 5))
            sns.barplot(x=month_counts.index, y=month_counts.values, color="steelblue")
            plt.title("Uber Trips by Month")
            plt.xlabel("Month")
            plt.ylabel("Number of Trips")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.savefig("trips_by_month.png")
            plt.close()

            # --- 2. Bar chart: Trips by Day of Week ---
            day_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
                         "Friday", "Saturday", "Sunday"]
            day_counts = self.df["DayOfWeek"].value_counts().reindex(day_order)
            plt.figure(figsize=(10, 5))
            sns.barplot(x=day_counts.index, y=day_counts.values, color="darkorange")
            plt.title("Uber Trips by Day of the Week")
            plt.xlabel("Day of Week")
            plt.ylabel("Number of Trips")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.savefig("trips_by_day.png")
            plt.close()

            # --- 3. Histogram: Trips by Hour ---
            plt.figure(figsize=(10, 5))
            sns.histplot(self.df["Hour"], bins=24, kde=False, color="seagreen")
            plt.title("Uber Trips by Hour of Day")
            plt.xlabel("Hour of Day")
            plt.ylabel("Number of Trips")
            plt.xticks(range(0, 24))
            plt.tight_layout()
            plt.savefig("trips_by_hour.png")
            plt.close()

            # --- 4. Heatmap: Day vs Hour frequency ---
            pivot = self.df.pivot_table(
                index="DayOfWeek", columns="Hour", values=self.datetime_col,
                aggfunc="count"
            ).reindex(day_order)
            pivot = pivot.fillna(0)

            plt.figure(figsize=(14, 6))
            sns.heatmap(pivot, cmap="YlOrRd", annot=False, linewidths=0.5)
            plt.title("Trip Frequency Heatmap (Day vs Hour)")
            plt.xlabel("Hour of Day")
            plt.ylabel("Day of Week")
            plt.tight_layout()
            plt.savefig("trip_heatmap.png")
            plt.close()

            print("\n[SUCCESS] Visualizations generated and saved as PNG files:")
            print(" - trips_by_month.png")
            print(" - trips_by_day.png")
            print(" - trips_by_hour.png")
            print(" - trip_heatmap.png")

        except Exception as e:
            print(f"\n[ERROR] An error occurred while generating visualizations: {e}")

In [ ]:
 def show_insights(self):
        """Display key insights derived from the analysis."""
        if not self._check_ready():
            return

        # Ensure all analyses have been run at least once to populate insights
        if "busiest_month" not in self.insights:
            self.analyze_by_month()
        if "busiest_day" not in self.insights:
            self.analyze_by_day()
        if "peak_hour" not in self.insights:
            self.analyze_by_hour()

        print("\n--- KEY INSIGHTS ---")
        print(f"Busiest Month : {self.insights.get('busiest_month')} "
              f"({self.insights.get('busiest_month_count')} trips)")
        print(f"Busiest Day   : {self.insights.get('busiest_day')} "
              f"({self.insights.get('busiest_day_count')} trips)")
        print(f"Peak Hour     : {self.insights.get('peak_hour')}:00 "
              f"({self.insights.get('peak_hour_count')} trips)")


In [ ]:
  def show_summary_report(self):
        """Display a final summary report of the entire analysis session."""
        print("\n" + "=" * 50)
        print("           UBER DATA ANALYSIS - SUMMARY REPORT")
        print("=" * 50)

        if self.df is None:
            print("No dataset was loaded during this session.")
            print("=" * 50)
            return

        print(f"Total Records Analyzed : {self.df.shape[0]}")
        print(f"Total Columns           : {self.df.shape[1]}")
        print(f"Data Cleaning Performed : {'Yes' if self.cleaned else 'No'}")

        if self.insights:
            print("\nKey Findings:")
            if "busiest_month" in self.insights:
                print(f"  - Busiest Month: {self.insights['busiest_month']}")
            if "busiest_day" in self.insights:
                print(f"  - Busiest Day  : {self.insights['busiest_day']}")
            if "peak_hour" in self.insights:
                print(f"  - Peak Hour    : {self.insights['peak_hour']}:00")
        else:
            print("\nNo analysis was performed during this session.")

        print("=" * 50)
        print("Thank you for using the Uber Data Analysis App!")
        print("=" * 50)

In [ ]:
  def _check_ready(self):
        """Verify that data has been loaded and cleaned before analysis."""
        if self.df is None:
            print("\n[ERROR] No dataset loaded. Please load a dataset first.")
            return False
        if not self.cleaned:
            print("\n[ERROR] Please perform data cleaning before running analysis.")
            return False
        if self.datetime_col is None or "Hour" not in self.df.columns:
            print("\n[ERROR] No valid datetime column was found/processed. "
                  "Time-based analysis cannot be performed.")
            return False
        return True


In [ ]:
import sys  # <--- FIXED: Added missing import for sys

# ---------------------------------------------------------------------------
# CLASS DEFINITION
# ---------------------------------------------------------------------------
# FIXME: Replace this skeleton with your actual UberDataAnalyzer implementation
# or import it if it lives in a different file.
class UberDataAnalyzer:
    def __init__(self):
        self.df = None

    def load_data(self, filepath):
        print(f"\n[INFO] Loading data from: {filepath} (Stub functionality)")
        # Your loading logic here

    def show_dataset_info(self):
        print("\n[INFO] Displaying dataset info... (Stub functionality)")

    def clean_data(self):
        print("\n[INFO] Cleaning data... (Stub functionality)")

    def analyze_by_month(self):
        print("\n[INFO] Analyzing trips by month... (Stub functionality)")

    def analyze_by_day(self):
        print("\n[INFO] Analyzing trips by day... (Stub functionality)")

    def analyze_by_hour(self):
        print("\n[INFO] Analyzing trips by hour... (Stub functionality)")

    def generate_visualizations(self):
        print("\n[INFO] Generating plots... (Stub functionality)")

    def show_insights(self):
        print("\n[INFO] Showing key insights... (Stub functionality)")

    def show_summary_report(self):
        print("\n[INFO] Showing final summary report... (Stub functionality)")


# ---------------------------------------------------------------------------
# MENU & USER INTERFACE
# ---------------------------------------------------------------------------
def print_menu():
    """Display the main menu options."""
    print("\n" + "-" * 50)
    print("           UBER DATA ANALYSIS APP - MAIN MENU")
    print("-" * 50)
    print("1. Load Dataset")
    print("2. View Dataset Information")
    print("3. Perform Data Cleaning")
    print("4. Analyze Trips by Month")
    print("5. Analyze Trips by Day")
    print("6. Analyze Trips by Hour")
    print("7. Generate Visualizations")
    print("8. Show Insights")
    print("9. Exit")
    print("-" * 50)


def main():
    """Main driver function implementing the menu-driven console loop."""
    analyzer = UberDataAnalyzer()  # Now works because the class exists!

    print("Welcome to the Uber Data Analysis App!")

    while True:
        print_menu()
        try:
            choice = input("Enter your choice (1-9): ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nExiting application. Goodbye!")
            break

        if choice == "1":
            filepath = input("Enter path to the CSV file: ").strip()
            analyzer.load_data(filepath)

        elif choice == "2":
            analyzer.show_dataset_info()

        elif choice == "3":
            analyzer.clean_data()

        elif choice == "4":
            analyzer.analyze_by_month()

        elif choice == "5":
            analyzer.analyze_by_day()

        elif choice == "6":
            analyzer.analyze_by_hour()

        elif choice == "7":
            analyzer.generate_visualizations()

        elif choice == "8":
            analyzer.show_insights()

        elif choice == "9":
            analyzer.show_summary_report()
            print("\nExiting application. Goodbye!")
            break

        else:
            print("\n[INVALID INPUT] Please enter a number between 1 and 9.")


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"\n[FATAL ERROR] The application encountered an unexpected error: {e}")
        sys.exit(1)

Welcome to the Uber Data Analysis App!

--------------------------------------------------
           UBER DATA ANALYSIS APP - MAIN MENU
--------------------------------------------------
1. Load Dataset
2. View Dataset Information
3. Perform Data Cleaning
4. Analyze Trips by Month
5. Analyze Trips by Day
6. Analyze Trips by Hour
7. Generate Visualizations
8. Show Insights
9. Exit
--------------------------------------------------
